# Análisis Exploratorio de Datos

Este notebook documenta el análisis exploratorio del dataset de visitantes web. El propósito no es solo describir lo que hay; es identificar los hechos sobre los datos que condicionan las decisiones de preprocesamiento y modelado posteriores.

**Autor:** Andrés Fernando Gómez Rojas

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis, spearmanr, chi2_contingency, kruskal
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

from src.data_loader import load_raw, validate_schema

pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.dpi'] = 110

## Carga y validación del esquema

La función `validate_schema` corre chequeos de coherencia lógica: que las proporciones estén en [0,1], que las horas estén entre 0 y 23, que los pageviews no superen los hits, y que cada visitante aparezca una sola vez.

In [2]:
df = load_raw('../data/raw/data_customers.csv')
report = validate_schema(df)
for k, v in report.items():
    if not isinstance(v, dict):
        print(f'{k:40s} {v}')

n_filas                                  9996
n_visitantes_unicos                      9996
una_fila_por_visitante                   True
duplicados_totales                       0
pageviews_menor_igual_hits               True
bounce_en_rango                          True
weekend_en_rango                         True
hour_en_rango                            True
sesiones_positivas                       True


### Hallazgo clave inicial

El dataset viene pre-agregado a nivel visitante. Hay 9.996 filas y 9.996 fullVisitorId únicos, una fila por persona. La columna `sessionId` no es un identificador sino el conteo de sesiones del visitante (rango 1 a 278). Renombramos a `n_sessions` para que el código refleje el significado real.

Las proporciones `weekend_prop`, `bounce_prop` y `device.isMobile` son promedios a través de las sesiones del visitante. Cuatro personas tienen valores intermedios en `isMobile`, lo que indica que cambiaron de dispositivo entre sesiones.

## Diccionario de datos y cardinalidad

In [3]:
diccionario = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(),
    'pct_unique': (df.nunique() / len(df) * 100).round(2),
    'nulos': df.isnull().sum(),
    'ejemplo': [df[c].dropna().iloc[0] for c in df.columns],
})
diccionario

,dtype,n_unique,pct_unique,nulos,ejemplo
fullVisitorId,uint64,9996,100.000,0,213131142648941
channelGrouping,str,8,0.080,0,Direct
weekend_prop,float64,104,1.040,0,0.000
hour,float64,47,0.470,0,22.000
n_sessions,int64,54,0.540,0,1
device.browser,str,9,0.090,0,Chrome
device.deviceCategory,str,3,0.030,0,desktop
device.isMobile,float64,6,0.060,0,0.000
device.operatingSystem,str,7,0.070,0,Macintosh
totals.hits,float64,235,2.350,0,14.000


## Análisis univariado numérico

Reportamos media, mediana, percentiles, skewness y kurtosis. El skewness mide asimetría de la cola; valores por encima de 1 indican cola derecha pronunciada. La kurtosis mide qué tan pesada es esa cola; valores muy por encima de 3 (la kurtosis de una normal) indican outliers extremos.

In [4]:
num_vars = ['n_sessions', 'totals.hits', 'totals.pageviews',
            'bounce_prop', 'weekend_prop', 'hour']
stats_univ = pd.DataFrame({
    'media': df[num_vars].mean(),
    'mediana': df[num_vars].median(),
    'p95': df[num_vars].quantile(0.95),
    'p99': df[num_vars].quantile(0.99),
    'max': df[num_vars].max(),
    'skewness': df[num_vars].apply(skew),
    'kurtosis': df[num_vars].apply(kurtosis),
})
stats_univ

,media,mediana,p95,p99,max,skewness,kurtosis
n_sessions,3.606,2.000,10.000,21.000,278.000,19.664,656.769
totals.hits,22.178,17.000,59.625,101.000,500.000,5.127,68.347
totals.pageviews,17.530,14.000,44.500,71.050,466.000,6.651,127.052
bounce_prop,0.083,0.000,0.500,0.667,0.925,1.973,3.095
weekend_prop,0.147,0.000,1.000,1.000,1.000,2.031,2.850
hour,14.449,16.500,22.000,23.000,23.000,-0.902,-0.318


Las tres variables de conteo (n_sessions, hits, pageviews) muestran skewness por encima de 5 y kurtosis enorme. La kurtosis de 657 en n_sessions confirma una cola extraordinariamente pesada. Esto justifica:

* Usar Spearman en lugar de Pearson para correlaciones, ya que Pearson asume normalidad.
* Usar Kruskal-Wallis en lugar de ANOVA para los tests posteriores entre clusters.
* Aplicar log + clip al percentil 99 antes del clustering basado en distancia euclidiana, para que los outliers no dominen las distancias.

## Detección de outliers univariados con IQR

In [5]:
for v in ['n_sessions', 'totals.hits', 'totals.pageviews', 'bounce_prop']:
    q1, q3 = df[v].quantile([0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    n_out = (df[v] > upper).sum()
    print(f'{v:25s}  outliers = {n_out:5d} ({n_out/len(df)*100:.1f}%)  umbral = {upper:.2f}')

n_sessions                 outliers =   689 (6.9%)  umbral = 8.50
totals.hits                outliers =   594 (5.9%)  umbral = 56.50
totals.pageviews           outliers =   501 (5.0%)  umbral = 44.25
bounce_prop                outliers =  1792 (17.9%)  umbral = 0.21


## Univariado categórico y desbalance

El índice de Gini de impureza mide qué tan repartida está una variable categórica entre sus posibles valores. Cuando una categoría domina, el Gini se acerca a cero.

In [6]:
def gini(s):
    p = s.value_counts(normalize=True).values
    return 1 - (p ** 2).sum()

cat_vars = ['channelGrouping', 'device.browser', 'device.deviceCategory',
            'device.operatingSystem', 'trafficSource.medium']
resumen_cat = pd.DataFrame({
    'cardinalidad': [df[v].nunique() for v in cat_vars],
    'top_categoria': [df[v].value_counts().index[0] for v in cat_vars],
    'top_pct': [df[v].value_counts(normalize=True).iloc[0] * 100 for v in cat_vars],
    'gini': [gini(df[v]) for v in cat_vars],
    'categorias_raras_lt_1pct': [(df[v].value_counts(normalize=True) < 0.01).sum()
                                  for v in cat_vars],
}, index=cat_vars)
resumen_cat

,cardinalidad,top_categoria,top_pct,gini,categorias_raras_lt_1pct
channelGrouping,8,Referral,42.877,0.675,3
device.browser,9,Chrome,89.406,0.195,6
device.deviceCategory,3,desktop,90.306,0.178,0
device.operatingSystem,7,Macintosh,55.532,0.636,1
trafficSource.medium,7,referral,43.667,0.673,2


Dominancias importantes: Chrome al 89%, Desktop al 90%, Macintosh al 56%, Referral al 43%. Es la huella típica del Google Merchandise Store, que históricamente se comparte como muestra de Google Analytics. Esto condiciona la interpretación de los segmentos: cualquier hallazgo sobre mobile aplica solo al 10% de la muestra.

## Correlaciones y multicolinealidad

Spearman es preferible a Pearson cuando los datos no son normales. Reportamos también el Variance Inflation Factor, que diagnostica multicolinealidad: valores por encima de 5 indican redundancia problemática.

In [7]:
spearman_mat = df[num_vars].corr(method='spearman').round(2)
print('Matriz Spearman:')
print(spearman_mat)
print('\nPares con |rho| > 0.5:')
for i in range(len(num_vars)):
    for j in range(i + 1, len(num_vars)):
        r = spearman_mat.iloc[i, j]
        if abs(r) > 0.5:
            print(f'  {num_vars[i]:20s} <-> {num_vars[j]:20s}  rho = {r:+.2f}')

Matriz Spearman:
                  n_sessions  totals.hits  totals.pageviews  bounce_prop  \
n_sessions             1.000       -0.640            -0.670        0.500   
totals.hits           -0.640        1.000             0.990       -0.470   
totals.pageviews      -0.670        0.990             1.000       -0.490   
bounce_prop            0.500       -0.470            -0.490        1.000   
weekend_prop           0.140       -0.070            -0.070        0.080   
hour                   0.020       -0.060            -0.060       -0.000   

                  weekend_prop   hour  
n_sessions               0.140  0.020  
totals.hits             -0.070 -0.060  
totals.pageviews        -0.070 -0.060  
bounce_prop              0.080 -0.000  
weekend_prop             1.000 -0.130  
hour                    -0.130  1.000  

Pares con |rho| > 0.5:
  n_sessions           <-> totals.hits           rho = -0.64
  n_sessions           <-> totals.pageviews      rho = -0.67
  totals.hits          <

In [8]:
X_vif = StandardScaler().fit_transform(df[num_vars])
vif_df = pd.DataFrame({
    'variable': num_vars,
    'VIF': [variance_inflation_factor(X_vif, i) for i in range(len(num_vars))]
}).sort_values('VIF', ascending=False)
vif_df

,variable,VIF
2,totals.pageviews,28.252
1,totals.hits,27.925
3,bounce_prop,1.128
0,n_sessions,1.088
5,hour,1.023
4,weekend_prop,1.018


`totals.hits` y `totals.pageviews` tienen VIF cercano a 28: redundancia severa. Conservamos las dos en el clustering porque cada una aporta señal levemente distinta, pero derivamos `hits_per_pageview` como medida de interacción no-página.

Spearman también revela algo contraintuitivo: la correlación entre `n_sessions` y `totals.hits` es **negativa**. Quien viene más veces tiene menos hits totales que quien viene pocas veces. La interpretación es que hay dos comportamientos opuestos: visitantes que vienen una o dos veces y hacen sesiones largas de research, y visitantes que vienen muchas veces pero hacen sesiones cortas de chequeo rápido. Esto anticipa la estructura que el clustering va a recuperar.

## Redundancia entre variables categóricas

In [9]:
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))

print('Pares con Cramer V > 0.3:')
seen = set()
for c1 in cat_vars:
    for c2 in cat_vars:
        if c1 != c2 and (c2, c1) not in seen:
            v = cramers_v(df[c1], df[c2])
            seen.add((c1, c2))
            if v > 0.3:
                lectura = 'REDUNDANTES' if v > 0.9 else 'fuerte' if v > 0.5 else 'moderada'
                print(f'  {c1:28s} <-> {c2:28s}  V = {v:.3f}  ({lectura})')

Pares con Cramer V > 0.3:
  channelGrouping              <-> trafficSource.medium          V = 0.983  (REDUNDANTES)
  device.browser               <-> device.deviceCategory         V = 0.344  (moderada)
  device.browser               <-> device.operatingSystem        V = 0.306  (moderada)
  device.deviceCategory        <-> device.operatingSystem        V = 0.708  (fuerte)


`channelGrouping` y `trafficSource.medium` tienen Cramer's V = 0.98: son prácticamente la misma variable. Eliminamos `trafficSource.medium` por redundancia perfecta. Esta decisión queda justificada con número, no con intuición.

## Outliers multivariados con Isolation Forest

Los outliers univariados no capturan combinaciones inusuales de variables. Un visitante con sesiones promedio pero engagement extremo puede ser un outlier multivariado aunque ninguna variable individual lo marque.

In [10]:
X_iso = StandardScaler().fit_transform(df[num_vars])
iso = IsolationForest(contamination=0.05, random_state=42).fit(X_iso)
df['is_outlier_iso'] = iso.predict(X_iso) == -1
perfil_out = df.groupby('is_outlier_iso')[num_vars].mean().round(2).T
perfil_out.columns = ['no_outlier', 'outlier']
perfil_out['ratio'] = (perfil_out['outlier'] / perfil_out['no_outlier']).round(2)
perfil_out

,no_outlier,outlier,ratio
n_sessions,3.330,8.800,2.640
totals.hits,20.320,57.510,2.830
totals.pageviews,16.230,42.190,2.600
bounce_prop,0.080,0.210,2.620
weekend_prop,0.130,0.500,3.850
hour,14.640,10.810,0.740


Los outliers multivariados son visitantes con engagement aproximadamente cinco veces superior al típico. El separador más fuerte entre outliers y resto es `weekend_prop`, lo que anticipa la existencia del cluster Weekend Warriors antes incluso de hacer clustering.

**Decisión sobre estos outliers:** no se eliminan. Son power users que el negocio quiere identificar precisamente. Sí los tratamos con log + clip al P99 antes del clustering para que no dominen las distancias.

## Síntesis del EDA

1. Dataset pre-agregado a nivel visitante, 9.996 filas, una por persona. La columna `sessionId` es conteo de sesiones, no identificador.
2. Calidad perfecta: cero nulos, cero duplicados, todas las restricciones lógicas se cumplen.
3. Variables de conteo extremadamente sesgadas. Requieren log-transform y clip P99 antes de modelos basados en distancia.
4. Multicolinealidad severa: hits y pageviews con VIF 28, channelGrouping y trafficSource.medium con Cramer V 0.98.
5. Dominancias: el dataset es Desktop/Chrome/Macintosh, con minoría mobile del 10%.
6. Correlación negativa Spearman entre n_sessions y hits totales, que anticipa al menos dos perfiles de comportamiento opuestos.
7. Cinco por ciento de outliers multivariados son power users; no se eliminan, se transforman.

Estas siete observaciones determinan completamente el preprocesamiento del siguiente notebook.